In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\DR.KSS_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,346.0,176.0,208.0,133.0,200.0,245.0,105.0,64.0,101.0,149.0,339.0,285.0
1,2,340.0,215.0,117.0,144.0,197.0,173.0,118.0,76.0,87.0,173.0,318.0,280.0
2,3,341.0,199.0,126.0,167.0,264.0,155.0,108.0,68.0,89.0,162.0,381.0,268.0
3,4,377.0,274.0,141.0,173.0,282.0,211.0,61.0,64.0,69.0,184.0,380.0,178.0
4,5,333.0,177.0,125.0,174.0,292.0,251.0,77.0,59.0,83.0,145.0,372.0,165.0
5,6,319.0,140.0,138.0,168.0,248.0,183.0,65.0,61.0,95.0,143.0,351.0,197.0
6,7,333.0,174.0,181.0,179.0,303.0,231.0,56.0,61.0,71.0,126.0,377.0,233.0
7,8,345.0,161.0,157.0,191.0,225.0,237.0,56.0,54.0,91.0,167.0,379.0,302.0
8,9,343.0,146.0,145.0,203.0,179.0,182.0,84.0,61.0,114.0,164.0,350.0,186.0
9,10,273.0,295.0,182.0,222.0,180.0,172.0,138.0,71.0,106.0,132.0,334.0,234.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,346.000000,176.000000,208.0,133.000000,200.0,245.000000,105.0,64.000000,101.000000,149.0,339.000000,285.0
1,2,340.000000,215.000000,117.0,144.000000,197.0,173.000000,118.0,76.000000,87.000000,173.0,318.000000,280.0
2,3,341.000000,199.000000,126.0,167.000000,264.0,155.000000,108.0,68.000000,89.000000,162.0,381.000000,268.0
3,4,377.000000,274.000000,141.0,173.000000,282.0,211.000000,61.0,64.000000,69.000000,184.0,380.000000,178.0
4,5,333.000000,177.000000,125.0,174.000000,292.0,251.000000,77.0,59.000000,83.000000,145.0,372.000000,165.0
5,6,319.000000,140.000000,138.0,168.000000,248.0,183.000000,65.0,61.000000,95.000000,143.0,351.000000,197.0
6,7,333.000000,174.000000,181.0,179.000000,303.0,231.000000,56.0,61.000000,71.000000,126.0,377.000000,233.0
7,8,345.000000,161.000000,157.0,191.000000,225.0,237.000000,56.0,54.000000,91.000000,167.0,379.000000,302.0
8,9,343.000000,146.000000,145.0,203.000000,179.0,182.000000,84.0,61.000000,114.000000,164.0,350.000000,186.0
9,10,273.000000,295.000000,182.0,222.000000,180.0,172.000000,138.0,71.000000,106.000000,132.0,334.000000,234.0
